# 실습 — 글자가 아니라 '의미'로 찾기

**위니브 카페 CS 챗봇 만들기 **



## 오늘 배울 것 

> 고객은 "환불"이라 쓰는데 문서엔 "반품"이라 적혀 있으면,
> 글자만 맞추는 검색은 실패합니다. **뜻으로 찾는 법**을 배웁니다.

| 단계 | 내용 |
|------|------|
| 1️⃣ 이론 | 글자 검색은 왜 실패하나? 의미 검색이란? |
| 2️⃣ 기본 | 문장을 숫자로 바꾸기 (임베딩) |
| 3️⃣ 응용 | 검색 개수·점수 다뤄보기 |
| 4️⃣ 실습 | 위니브 문서를 의미로 검색하기 |


# 1️⃣ 단계 — 이론 

##  만든 것

위니브 문서를 작은 **청크(조각)** 로 잘라뒀습니다.
이제 진짜 질문이 남았습니다.

> 🤔 고객이 "환불할래요"라고 물으면, 수십 개 청크 중에서
> **어느 조각이 답인지** 컴퓨터가 어떻게 찾을까?

## 글자만 맞추는 검색의 한계

가장 단순한 방법은 "질문에 든 단어가 들어간 조각 찾기"입니다.
(Ctrl+F 와 같음 — 이걸 키워드 검색이라 합니다.) 그런데:

> 🧑 고객: "**환불**할래요"
> 📄 문서: "단순 변심 **반품**은 7일 이내…"

사람은 "환불"과 "반품"이 같은 뜻인 걸 압니다. 하지만 컴퓨터에게
글자는 그냥 기호라 `환불 ≠ 반품` 입니다. 답이 문서에 있는데도
**검색이 실패**합니다.

| 고객이 쓰는 말 | 문서에 적힌 말 | 글자 검색 |
|----------------|----------------|-----------|
| 환불할래요 | 반품은 7일 이내 | ❌ 못 찾음 |
| 포인트 언제 사라져요 | 적립금 유효기간 1년 | ❌ 못 찾음 |

> 우리가 원하는 건: **"글자가 달라도 뜻이 통하면 찾아 줘."**

## 해결책 ① 임베딩 — 문장을 '숫자'로 바꾼다

**임베딩(Embedding)** = 문장의 **뜻**을 숫자 묶음(좌표)으로 바꾸는 것.

> 💡 비유 — 의미의 지도:
> 도시를 좌표로 표시하면 서울-인천은 가깝고, 서울-부산은 멀죠.
> 임베딩은 이걸 *문장의 뜻* 에 적용합니다.
>
> ```
> "환불할래요" ──●
>                 ╲ 아주 가까움 (뜻이 비슷)
> "반품 신청"  ───●
> "배송비 얼마" ────────────● 멀다 (뜻이 다름)
> ```
>
> 뜻이 비슷한 문장은 좌표가 **가깝게** 됩니다. 그러면 컴퓨터도
> "환불"과 "반품"이 가깝다는 걸 계산으로 알 수 있습니다.

## 해결책 ② 벡터 DB — 가까운 걸 빨리 찾는 창고

청크가 수천 개면 매번 전부 비교하면 느립니다. 그래서 좌표들을
잘 정리해두고 "가까운 것"을 빠르게 찾아주는 창고가 필요합니다.
이게 **벡터 데이터베이스** 입니다. (입문용  **Chroma** 사용)

> 💡 비유 — 정리된 도서관 서가: 비슷한 책을 가까운 칸에 꽂아두면
> 한 칸만 봐도 관련 책이 쭉 있죠. 벡터 DB가 그 서가입니다.

**전체 그림**:
```
[준비] 청크 → 임베딩(숫자로) → 벡터 DB에 저장
[검색] 질문 → 임베딩(숫자로) → 벡터 DB에서 가까운 청크 꺼내기
```

---
👉 아래부터 코드입니다. 한 줄씩 천천히 갑니다.


# 2️⃣ 단계 — 문장을 숫자로 바꿔보기

"임베딩 = 문장을 숫자로" 가 진짜인지 직접 확인합니다.

### 셀 1 — 도구 설치

임베딩 모델과 벡터 DB(Chroma) 도구를 설치합니다.

In [1]:
!pip install -q langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters langchain-huggingface python-dotenv sentence-transformers

### 셀 2 — 임베딩 모델 준비 (키 없어도 OK!)

문장을 숫자로 바꿔주는 **임베딩 모델**을 준비합니다.
- OpenAI 키가 있으면 → OpenAI 모델 (빠름)
- 키가 없으면 → **무료 모델** 자동 사용

어느 쪽이든 다음 셀부터는 똑같이 진행됩니다.

In [2]:
import os

# 키가 있으면 환경에 등록 (Colab 보안비밀 / 로컬 .env)
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

# 키 유무에 따라 임베딩 모델 자동 선택
if os.environ.get("OPENAI_API_KEY"):
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("✅ OpenAI 임베딩 사용")
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    print("✅ 무료 임베딩 사용 (처음 한 번 다운로드, 1~2분)")

✅ OpenAI 임베딩 사용


**이 셀이 한 일**: 문장을 숫자로 바꿔줄 임베딩 모델을
준비했습니다. 키가 없어도 무료 모델로 자동 전환됩니다.

### 셀 3 — 문장이 진짜 숫자가 되는지 보기

문장 하나를 임베딩 모델에 넣어봅니다. 숫자 묶음이 나옵니다.

In [3]:
vector = embeddings.embed_query("환불 규정 알려줘")

print("숫자가 몇 개?:", len(vector), "개")
print("앞 5개만 보기:", [round(x, 3) for x in vector[:5]])

숫자가 몇 개?: 1536 개
앞 5개만 보기: [-0.012, 0.029, -0.025, 0.02, -0.061]


**이 셀이 한 일**: "환불 규정 알려줘" 라는 문장이 여러 개의
숫자로 바뀌었습니다. 이 숫자 묶음이 바로 그 문장의 **의미 좌표**
입니다.

### 셀 4 — '가까움'을 숫자로 확인 

뜻이 비슷한 문장은 좌표가 가깝습니다. 두 문장이 얼마나 가까운지
점수로 재봅니다. (1에 가까울수록 비슷)

In [4]:
import numpy as np

def similarity(text1, text2):
    v1 = np.array(embeddings.embed_query(text1))
    v2 = np.array(embeddings.embed_query(text2))
    # 두 좌표가 얼마나 같은 방향인지 (1=매우 비슷, 0=무관)
    return float(v1 @ v2 / (np.linalg.norm(v1) * np.linalg.norm(v2)))

print("'환불할래요' ↔ '반품 신청'  :", round(similarity("환불할래요", "반품 신청"), 3))
print("'환불할래요' ↔ '배송비 얼마':", round(similarity("환불할래요", "배송비 얼마"), 3))

'환불할래요' ↔ '반품 신청'  : 0.348
'환불할래요' ↔ '배송비 얼마': 0.301


**이 셀이 한 일**: 두 문장의 의미가 얼마나 가까운지 쟀습니다.

> 💡 '환불할래요 ↔ 반품 신청' 점수가 '환불할래요 ↔ 배송비'보다
> **훨씬 높게** 나옵니다. 글자는 다른데 뜻이 가까우니 숫자도
> 가까운 것 — 이게 의미 검색이 가능한 이유입니다!

# 3️⃣ 단계 — 벡터 DB에 저장하고 검색하기

문장을 숫자로 바꿀 줄 알았으니, 이제 여러 문장을 저장해두고
"가까운 것"을 검색해 봅니다.

### 셀 5 — 연습용 문장들을 벡터 DB에 저장

작은 예제 문장 3개를 Chroma 벡터 DB에 넣습니다.

In [5]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

# 연습용 문장 3개 (각자 분류 꼬리표를 달아둠)
sample_docs = [
    Document(page_content="환불은 수령 후 7일 이내 신청 가능합니다.", metadata={"topic": "refund"}),
    Document(page_content="배송비는 3,000원, 5만원 이상 무료입니다.", metadata={"topic": "shipping"}),
    Document(page_content="회원 등급은 누적 구매액으로 결정됩니다.", metadata={"topic": "membership"}),
]

# 벡터 DB에 저장 (문장을 자동으로 숫자로 바꿔서 저장함)
db = Chroma.from_documents(sample_docs, embeddings)
print("벡터 DB에 문장 3개 저장 완료 ✅")

벡터 DB에 문장 3개 저장 완료 ✅


**이 셀이 한 일**: 문장 3개를 벡터 DB에 저장했습니다.
저장할 때 자동으로 임베딩(숫자 변환)이 일어납니다.

### 셀 6 — 의미로 검색 (글자가 달라도 찾는다!)

"돈 언제 돌려받아요?" 로 검색합니다. 이 문장엔 '환불'이라는
글자가 **없습니다.** 그래도 환불 문장을 찾을까요?

In [6]:
result = db.similarity_search("돈 언제 돌려받아요?", k=2)

print("질문: 돈 언제 돌려받아요?")
print("찾은 문장:", result[0].page_content)
print("분류:", result[0].metadata["topic"])

질문: 돈 언제 돌려받아요?
찾은 문장: 회원 등급은 누적 구매액으로 결정됩니다.
분류: membership


**이 셀이 한 일**: '회원'이라는 글자가 없는 질문인데도
**맴버십 문장**을 찾아냈습니다.

> 💡 글자 검색이었다면 절대 못 찾았을 질문입니다. 의미(좌표)로
> 찾으니 "돈 돌려받기 = 맴버십, 등급"을 연결한 것입니다.

### 셀 7 — 검색 개수(k) 바꿔보기

`k` 는 "가까운 것 몇 개를 가져올지"입니다. 바꿔서 비교합니다.

In [7]:
query = "환불 관련 알려줘"

for k in [1, 2, 3]:
    results = db.similarity_search(query, k=k)
    topics = [r.metadata["topic"] for r in results]
    print(f"k={k} → 가져온 분류: {topics}")

k=1 → 가져온 분류: ['refund']
k=2 → 가져온 분류: ['refund', 'shipping']
k=3 → 가져온 분류: ['refund', 'shipping', 'membership']


**이 셀이 한 일**: k를 1, 2, 3으로 바꿔 검색했습니다.

> 💡 k가 클수록 더 많이 가져오지만, 관련 약한 것(shipping,
> membership)도 섞이기 시작합니다. 보통 3~5에서 시작합니다.

### 셀 8 — 분류로 검색 범위 좁히기 (filter)

"환불 문장만 보고 싶다"면, 꼬리표(topic)로 검색 범위를 좁힙니다.
5교시에서 깊게 배울 기능을 미리 맛봅니다.

In [8]:
query = "관련 규정 알려줘"

print("───[ 전체에서 검색 ]───")
for r in db.similarity_search(query, k=2):
    print(" -", r.metadata["topic"], ":", r.page_content)

print("\n───[ refund 분류만 검색 ]───")
for r in db.similarity_search(query, k=2, filter={"topic": "refund"}):
    print(" -", r.metadata["topic"], ":", r.page_content)

───[ 전체에서 검색 ]───
 - membership : 회원 등급은 누적 구매액으로 결정됩니다.
 - refund : 환불은 수령 후 7일 이내 신청 가능합니다.

───[ refund 분류만 검색 ]───
 - refund : 환불은 수령 후 7일 이내 신청 가능합니다.


**이 셀이 한 일**: 필터를 걸면 `refund` 분류 문장만 검색
대상이 됩니다.

> 📌 정리: 임베딩으로 문장을 숫자로 → 벡터 DB에 저장 →
> `similarity_search` 로 의미 검색. `k`(개수)와 `filter`(분류)로
> 조절합니다.

# 4️⃣ 단계 — 위니브 문서를 의미로 검색하기

이제 진짜 위니브 문서 5개를 벡터 DB에 넣고, 글자 검색이
실패하던 질문을 의미 검색으로 해결합니다.

### 셀 9 — 위니브 문서 준비 (2교시 내용 압축)

2교시에서 배운 "불러오기 → 쪼개기 → 분류 꼬리표"를 한 셀에
모았습니다. (이미 배운 내용이라 빠르게 지나갑니다)

In [9]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = {
    "01_faq.md":        "# 위니브 카페 자주 묻는 질문 (FAQ)\n\n스페셜티 커피 전문 쇼핑몰 위니브 카페 고객센터입니다.\n자주 묻는 질문을 모았습니다.\n\n---\n\n### Q. 주문은 어떻게 하나요?\nA. 원하시는 상품을 장바구니에 담은 뒤 결제 페이지에서 배송지와 결제수단을\n입력하시면 됩니다. 회원·비회원 모두 주문 가능합니다.\n\n### Q. 비회원도 주문할 수 있나요?\nA. 네, 비회원 주문이 가능합니다. 다만 비회원 주문은 적립금이 적립되지 않으며,\n회원 등급 혜택도 적용되지 않습니다.\n\n### Q. 주문을 취소하고 싶어요.\nA. 상품이 발송되기 전이라면 마이페이지 > 주문내역에서 직접 취소할 수 있습니다.\n이미 발송된 경우에는 반품 절차로 진행됩니다. 자세한 내용은 환불·반품 정책을\n확인해 주세요.\n\n### Q. 결제 수단은 무엇이 있나요?\nA. 신용·체크카드, 계좌이체, 간편결제(카카오페이·네이버페이)를 지원합니다.\n무통장입금은 현재 지원하지 않습니다.\n\n### Q. 선물 포장이 되나요?\nA. 네, 가능합니다. 주문 시 '선물포장' 옵션을 선택하시면 건당 2,000원에\n포장해 드립니다. 선물 메시지 카드도 함께 넣어드립니다.\n\n### Q. 주문한 상품은 언제 받을 수 있나요?\nA. 평일 오후 2시 이전 결제 건은 당일 발송되며, 보통 발송 후 1~2일 이내\n도착합니다. 지역에 따라 더 걸릴 수 있습니다. 배송 정책 문서를 참고해 주세요.\n\n### Q. 배송 조회는 어디서 하나요?\nA. 마이페이지 > 주문내역에서 송장번호와 배송 상태를 확인할 수 있습니다.\n발송 완료 시 알림 문자도 발송됩니다.\n\n### Q. 원두를 분쇄해서 보내주나요?\nA. 네, 주문 시 분쇄도(홀빈·드립·에스프레소 등)를 선택하실 수 있습니다.\n신선도를 위해 홀빈(분쇄 안 함) 구매를 권장드립니다.\n\n### Q. 재고가 없는 상품은 언제 다시 들어오나요?\nA. 품절 상품 페이지에서 '재입고 알림'을 신청하시면 입고 시 문자로\n안내드립니다. 원두는 로스팅 일정에 따라 보통 주 1회 입고됩니다.\n\n### Q. 영수증·세금계산서 발급이 되나요?\nA. 현금영수증은 결제 시 신청 가능하며, 세금계산서는 사업자 회원에 한해\n고객센터로 요청하시면 발급해 드립니다.\n\n### Q. 상품 문의는 어디로 하나요?\nA. 상품 상세페이지의 Q&A 게시판 또는 고객센터 채팅으로 문의해 주세요.\n운영 시간은 평일 오전 10시~오후 5시입니다.",
    "02_refund.md":     "# 위니브 카페 환불·반품·교환 정책\n\n본 정책은 전자상거래법을 따르며, 위니브 카페에서 구매한 모든 상품에 적용됩니다.\n\n## 1. 반품·환불 신청 기간\n\n- **단순 변심**: 상품 수령일로부터 7일 이내 신청 가능\n- **상품 하자·오배송**: 수령일로부터 14일 이내 신청 가능\n- 위 기간이 지나면 반품·환불이 제한될 수 있습니다.\n\n## 2. 환불 불가 품목 (중요)\n\n다음 품목은 단순 변심에 의한 반품·환불이 **불가**합니다.\n\n- **개봉한 원두 및 드립백**: 식품 위생상 개봉 시 재판매가 불가능합니다.\n  미개봉 상태에서는 7일 이내 반품 가능합니다.\n- **디저트류(케이크·쿠키·구움과자 등)**: 신선식품으로, 단순 변심 반품이\n  불가합니다. 단, 변질·파손 등 하자가 있는 경우 수령 14일 이내 교환·환불됩니다.\n- **주문 제작 굿즈(각인·커스텀 텀블러 등)**: 고객 요청으로 제작된 상품은\n  단순 변심 반품이 불가합니다.\n- 단, 위 품목도 **상품 하자나 오배송**인 경우에는 환불·교환이 가능합니다.\n\n## 3. 단순 변심 반품 시 비용\n\n- 단순 변심으로 반품할 경우 왕복 배송비(편도 3,000원 × 2 = 6,000원)는\n  고객이 부담합니다.\n- 상품 하자·오배송으로 인한 반품은 배송비를 위니브 카페가 전액 부담합니다.\n\n## 4. 환불 처리 절차 및 기간\n\n1. 마이페이지 > 주문내역에서 반품 신청\n2. 반품 상품 발송 (회수 택배 신청 또는 직접 발송)\n3. 반품 상품 도착 후 상태 확인\n4. 확인 완료 후 **영업일 기준 3일 이내** 환불 처리\n\n- 카드 결제는 카드사 정책에 따라 취소 반영까지 영업일 3~5일이 추가로\n  소요될 수 있습니다.\n- 적립금으로 결제한 부분은 적립금으로 환원됩니다.\n\n## 5. 교환\n\n- 동일 상품의 색상·옵션 변경은 재고가 있는 경우 교환 가능합니다.\n- 하자 상품의 교환 배송비는 위니브 카페가 부담합니다.\n- 단순 변심에 의한 교환 배송비는 고객이 부담합니다.\n- 디저트류는 교환이 어려워 환불로 처리될 수 있습니다.\n\n## 6. 주문 취소\n\n- 상품 발송 **전**에는 마이페이지에서 즉시 취소 가능하며 전액 환불됩니다.\n- 발송 **후**에는 취소가 불가하며, 수령 후 반품 절차로 진행됩니다.\n\n## 7. 환불이 제한되는 경우\n\n- 고객의 사용·훼손으로 상품 가치가 현저히 감소한 경우\n- 개봉 후 상태인 식품(원두·드립백·디저트)의 단순 변심\n- 반품 기간(7일/14일)을 경과한 경우\n- 위 사유라도 상품 자체 하자가 입증되면 환불 대상이 됩니다.",
    "03_shipping.md":   "# 위니브 카페 배송 정책\n\n## 1. 배송비\n\n| 구분 | 배송비 |\n|------|--------|\n| 기본 배송비 | 3,000원 |\n| 5만원 이상 구매 | 무료 |\n| 제주 지역 | 기본 배송비 + 3,000원 |\n| 도서산간 지역 | 기본 배송비 + 3,000원 |\n\n- 5만원 이상 구매 시 기본 배송비는 무료이나, 제주·도서산간 추가비\n  (3,000원)는 별도로 부과됩니다.\n\n## 2. 배송 기간\n\n- 평일 **오후 2시 이전** 결제 완료 건은 당일 발송됩니다.\n- 오후 2시 이후 결제 건은 다음 영업일에 발송됩니다.\n- 발송 후 도착까지 보통 1~2일 소요됩니다.\n- 제주·도서산간은 2~4일이 소요될 수 있습니다.\n- 주말·공휴일은 발송되지 않습니다.\n\n## 3. 원두 로스팅 일정\n\n- 원두는 신선도를 위해 **주문 후 로스팅(주문 로스팅)** 으로 발송됩니다.\n- 로스팅은 매주 화·금요일에 진행되며, 해당 요일 직전 주문이 함께\n  발송됩니다. 이로 인해 원두 단일 주문은 최대 3~4일이 소요될 수 있습니다.\n- 원두와 일반 상품을 함께 주문하면 로스팅 일정에 맞춰 합포장 발송됩니다.\n\n## 4. 배송 조회\n\n- 마이페이지 > 주문내역에서 송장번호로 실시간 조회가 가능합니다.\n- 발송 완료 시 등록된 휴대폰으로 알림 문자가 발송됩니다.\n\n## 5. 배송 관련 안내\n\n- 택배사는 일반 택배(CJ대한통운)를 이용합니다.\n- **새벽배송·당일배송 서비스는 운영하지 않습니다.**\n- 부재 시 택배사 정책에 따라 경비실 위탁 또는 재배송됩니다.\n- 배송지 오기입으로 인한 반송 시 재발송 배송비는 고객이 부담합니다.\n\n## 6. 해외 배송\n\n- 현재 해외 배송은 지원하지 않습니다. 국내 배송만 가능합니다.",
    "04_catalog.md":    "# 위니브 카페 상품 정보 카탈로그\n\n각 상품에는 `카테고리` 정보가 있습니다. (메타데이터 필터 실습에 사용)\n카테고리: 원두 / 드립백 / 디저트 / 홈카페기기 / 바리스타굿즈\n\n---\n\n## [원두] 에티오피아 예가체프 G1\n\n- 카테고리: 원두\n- 가격: 18,000원 (250g)\n- 설명: 자스민·베르가못 향이 도는 밝은 산미의 워시드 원두입니다.\n- 보관: 직사광선을 피해 서늘한 곳에 보관, 개봉 후 2주 이내 소비 권장\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [원두] 브라질 산토스 NY2\n\n- 카테고리: 원두\n- 가격: 14,000원 (250g)\n- 설명: 견과류·다크초콜릿의 묵직한 바디. 에스프레소 베이스로 인기.\n- 보관: 밀폐 용기 보관, 개봉 후 2주 이내 소비 권장\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [드립백] 데일리 블렌드 드립백 10개입\n\n- 카테고리: 드립백\n- 가격: 9,000원 (10개입)\n- 설명: 사무실·여행용 휴대 간편 드립백. 균형 잡힌 블렌드.\n- 보관: 개별 포장, 실온 보관\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [디저트] 클래식 파운드 케이크\n\n- 카테고리: 디저트\n- 가격: 12,000원\n- 설명: 버터 풍미가 진한 수제 파운드 케이크. 커피와 페어링 추천.\n- 보관: 냉장 보관, 수령 후 3일 이내 섭취 권장\n- 환불: 신선식품으로 단순 변심 반품 불가 / 변질·파손 시 14일 이내 교환·환불\n\n## [디저트] 비스코티 세트 (6개)\n\n- 카테고리: 디저트\n- 가격: 9,500원\n- 설명: 아몬드·헤이즐넛 두 가지 맛 구움과자 세트.\n- 보관: 실온 보관, 수령 후 7일 이내 섭취 권장\n- 환불: 신선식품으로 단순 변심 반품 불가 / 하자 시 14일 이내 교환·환불\n\n## [홈카페기기] 핸드드립 세트 (드리퍼+서버)\n\n- 카테고리: 홈카페기기\n- 가격: 38,000원\n- 설명: 입문용 드리퍼와 600ml 유리 서버 구성.\n- 보관: -\n- 환불: 미사용 7일 이내 가능 / 하자 시 14일 이내 교환\n\n## [홈카페기기] 전동 그라인더 GR-200\n\n- 카테고리: 홈카페기기\n- 가격: 89,000원\n- 설명: 40단계 분쇄도 조절 코니컬 버 그라인더.\n- 보관: -\n- 환불: 미사용 7일 이내 가능 / 하자(작동 불량) 시 14일 이내 무상 교환\n\n## [바리스타굿즈] 위니브 로고 머그컵\n\n- 카테고리: 바리스타굿즈\n- 가격: 12,000원\n- 설명: 300ml 도자기 머그. 위니브 카페 로고 인쇄.\n- 보관: -\n- 환불: 미사용 7일 이내 가능\n\n## [바리스타굿즈] 커스텀 각인 텀블러 (주문제작)\n\n- 카테고리: 바리스타굿즈\n- 가격: 28,000원\n- 설명: 원하는 문구를 각인해 제작하는 보온 텀블러.\n- 보관: -\n- 환불: **주문 제작 상품으로 단순 변심 반품 불가** / 하자 시 교환 가능",
    "05_membership.md": "# 위니브 카페 멤버십·적립 안내\n\n## 1. 회원 등급\n\n회원 등급은 **최근 6개월 누적 구매액**을 기준으로 매월 1일 자동 산정됩니다.\n\n| 등급 | 최근 6개월 누적 구매액 | 적립률 |\n|------|----------------------|--------|\n| 베이직 | 0원 ~ | 1% |\n| 실버 | 10만원 이상 | 2% |\n| 골드 | 30만원 이상 | 3% |\n| VIP | 60만원 이상 | 5% |\n\n- 등급은 매월 1일 재산정되며, 기준 미달 시 하향될 수 있습니다.\n- 신규 가입 시 기본 등급은 베이직입니다.\n\n## 2. 적립금\n\n- 구매 확정 시 등급별 적립률에 따라 적립됩니다.\n- 적립금은 **적립일로부터 1년** 경과 시 자동 소멸됩니다.\n- 적립금은 1,000원 이상부터 사용 가능하며, 1원 단위로 사용할 수 있습니다.\n- 비회원 주문은 적립이 적용되지 않습니다.\n\n## 3. 신규 가입 혜택\n\n- 신규 회원가입 시 3,000원 웰컴 적립금이 즉시 지급됩니다.\n- 웰컴 적립금의 유효기간은 지급일로부터 30일입니다.\n\n## 4. 생일 쿠폰\n\n- 생일이 등록된 회원에게 생일月 1일에 10% 할인 쿠폰이 발급됩니다.\n- 생일 쿠폰은 발급일로부터 그 달 말일까지 사용 가능합니다.\n- 일부 할인 상품에는 중복 적용되지 않습니다.\n\n## 5. 적립금 환불 처리\n\n- 적립금으로 결제한 주문을 환불할 경우, 사용한 적립금은 적립금으로\n  환원됩니다(현금 환불 아님).\n- 환원된 적립금의 유효기간은 원래 적립금의 잔여 기간을 따릅니다.\n\n## 6. 등급 혜택 요약\n\n- 골드 등급 이상: 무료 선물포장 제공\n- VIP 등급: 신상품 우선 구매 및 분기별 시음 원두 증정",
}
os.makedirs("winib_docs", exist_ok=True)
for filename, content in documents.items():
    with open(f"winib_docs/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

# 불러오기 → 쪼개기
all_docs = DirectoryLoader("winib_docs", glob="*.md",
                           loader_cls=TextLoader,
                           loader_kwargs={"encoding": "utf-8"}).load()
chunks = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=30).split_documents(all_docs)

# 분류 꼬리표 달기
category_map = {"01_faq.md": "faq", "02_refund.md": "refund",
                "03_shipping.md": "shipping", "04_catalog.md": "catalog",
                "05_membership.md": "membership"}
for c in chunks:
    c.metadata["category"] = category_map[os.path.basename(c.metadata["source"])]

print("준비된 청크 개수:", len(chunks))

준비된 청크 개수: 25


C:\Users\user\AppData\Local\Temp\ipykernel_4688\2647565195.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


**이 셀이 한 일**: 위니브 문서 5개를 청크로 만들고 분류
꼬리표까지 달았습니다. (2교시에서 배운 그대로)

### 셀 10 — 위니브 청크를 벡터 DB에 저장

청크들을 벡터 DB에 넣습니다. (무료 모델은 1~2분 걸릴 수 있어요)

In [10]:
winib_db = Chroma.from_documents(chunks, embeddings)
print("위니브 청크를 벡터 DB에 저장 완료 ✅")
print("저장된 청크 수:", winib_db._collection.count())

위니브 청크를 벡터 DB에 저장 완료 ✅
저장된 청크 수: 28


**이 셀이 한 일**: 위니브 청크 전부를 의미 검색이 가능한
벡터 DB에 저장했습니다.

### 셀 11 — 글자 검색이 실패하던 질문 (핵심!)

"포인트 언제 사라져요?" — '포인트'는 위니브 문서에 거의 없는
단어입니다(문서엔 '적립금'이라 적혀 있음). 의미로 찾을까요?

In [11]:
query = "포인트 언제 사라져요?"
results = winib_db.similarity_search(query, k=3)

print("질문:", query)
print()
for i, r in enumerate(results, 1):
    filename = os.path.basename(r.metadata["source"])
    print(f"[{i}] (출처: {filename}, 분류: {r.metadata['category']})")
    print("   ", r.page_content[:70].replace("\n", " "), "...")

질문: 포인트 언제 사라져요?

[1] (출처: 04_catalog.md, 분류: catalog)
    ## [디저트] 비스코티 세트 (6개)  - 카테고리: 디저트 - 가격: 9,500원 - 설명: 아몬드·헤이즐넛 두 가지 맛  ...
[2] (출처: 01_faq.md, 분류: faq)
    ### Q. 선물 포장이 되나요? A. 네, 가능합니다. 주문 시 '선물포장' 옵션을 선택하시면 건당 2,000원에 포장해 드 ...
[3] (출처: 05_membership.md, 분류: membership)
    # 위니브 카페 멤버십·적립 안내  ## 1. 회원 등급  회원 등급은 **최근 6개월 누적 구매액**을 기준으로 매월 1일  ...


**이 셀이 한 일**: '포인트'라는 단어가 거의 없는데도
**멤버십 적립금 청크**를 찾아냈습니다.

> 💡 글자 검색이었다면 실패했을 질문입니다. 검색 결과에 
> 단 출처·분류 꼬리표가 따라온 것도 확인하세요. 

### 셀 12 — 글자 검색 vs 의미 검색 (직접 비교)

구어체 질문으로 두 방식을 비교합니다.

In [ ]:
q = "물건을 변경하고 싶은데 돈 돌려받을 수 있어요?"

# (1) 글자 검색 흉내: '환불' 또는 '반품' 글자가 든 청크 찾기
keyword_hits = [c for c in chunks if "환불" in c.page_content or "반품" in c.page_content]
print("───[ 글자 검색 ]───")
print(f"'환불/반품' 글자가 든 청크는 {len(keyword_hits)}개 있지만,")
print("질문에는 그 단어가 없어서 → 연결 못 함 ❌")

# (2) 의미 검색
print("\n───[ 의미 검색 ]───")
for r in winib_db.similarity_search(q, k=2):
    print(" -", r.page_content[:55].replace("\n", " "), "...")

───[ 글자 검색 ]───
'환불/반품' 글자가 든 청크는 14개 있지만,
질문에는 그 단어가 없어서 → 연결 못 함 ❌

───[ 의미 검색 ]───
 - ## 3. 단순 변심 반품 시 비용  - 단순 변심으로 반품할 경우 왕복 배송비(편도 3,000원  ...
 - - **개봉한 원두 및 드립백**: 식품 위생상 개봉 시 재판매가 불가능합니다.   미개봉 상태에서 ...


: 

**이 셀이 한 일**: 같은 질문을 두 방식으로 검색했습니다.

> - 글자 검색: 질문에 '환불/반품'이 없어 연결 자체가 안 됨
> - 의미 검색: 구어체인데도 환불 규정 청크를 찾아냄
>
> 고객은 규정집처럼 말하지 않습니다. 그래서 의미 검색이 필수입니다.

---
# 🎓 배운 것 정리

| 단계 | 한 일 | 핵심 |
|------|-------|------|
| 1️⃣ 이론 | 글자 검색 한계 → 의미 검색 | "가까우면 비슷한 뜻" |
| 2️⃣ 기본 | 문장 → 숫자(임베딩) | embed + 유사도 |
| 3️⃣ 응용 | 저장·검색·필터 | similarity_search |
| 4️⃣ 실습 | 위니브 의미 검색 | 글자로 못 찾던 질문 해결 |

**한 문장 요약**: 문장을 **숫자(좌표)** 로 바꿔 저장하면, 글자가
달라도 **뜻이 가까운** 청크를 찾을 수 있다.

**다음 시간(4교시)**: 찾아온 청크를 AI에게 줘서, 드디어 **출처까지
붙은 진짜 RAG 챗봇**을 완성합니다.

---
## 🤔 생각해보기 

셀 11에서 우리는 질문으로 "가까운 청크 3개"를 가져왔습니다.
그런데 아직 **고객에게 줄 답변 문장**은 안 만들었습니다.
검색 결과는 그냥 문서 조각일 뿐입니다.

1. 검색해 온 청크 3개를 고객이 그대로 보면 어떨까요?
   왜 한 번 더 처리가 필요할까요?
2. 검색이 가끔 관련 없는 청크를 섞어 가져온다면(셀 7에서 k를
   키웠을 때처럼), 그걸 그대로 AI에게 주면 어떤 위험이?
   (1교시의 '거짓말'을 떠올려보세요)
3. "검색된 청크"와 "고객 질문"을 어떻게 조합해 AI에게 줘야
   정확하고 출처 있는 답이 나올까요? 

> ✍️ 메모:
> - 1번:
> - 2번:
> - 3번:
